# 01 — PySpark lineage with Spline

Reads the 100-row NYC Yellow Taxi sample, transforms it via the
DataFrame API, and writes Parquet sinks so Spline captures full
column-level lineage. Iceberg tables are optional analytics artifacts
with weaker lineage visibility.

Inspect results at <http://localhost:9090>.

## Lineage cheat sheet

| Your transformation | Spline node to click | Column to inspect |
|--------------------|----------------------|-------------------|
| `UNIX_TIMESTAMP` diff | `Project` | `trip_minutes` |
| `GROUP BY` + `SUM` | `Aggregate` | `revenue`, `trips` |
| Column select | `Project` | dropped columns absent in output |

In Spline UI: Execution Plan → turn off Compact view → click the node →
Output Schema → select a column → Lineage.


In [ ]:
from pyspark.sql import functions as F
from _shared.spark_session import get_spark, SAMPLE_CSV, PARQUET_SINK

spark = get_spark()
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)


In [ ]:
taxi_pdf = spark.read.csv(SAMPLE_CSV, header=True, inferSchema=True)
print('row count:', taxi_pdf.count())
taxi_pdf.printSchema()
taxi_pdf.show(5, truncate=False)


## Build the analytics layers

Two derived frames:

* `trip_durations` — adds a duration column and keeps numeric metrics.
* `zone_revenue` — aggregates pickup-zone revenue.


In [ ]:
trip_durations = (
    taxi_pdf
    .withColumn(
        'trip_minutes',
        (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 60.0,
    )
    .select(
        'PULocationID', 'DOLocationID', 'payment_type',
        'trip_distance', 'trip_minutes', 'fare_amount', 'tip_amount', 'total_amount',
    )
)
trip_durations.show(5, truncate=False)

zone_revenue = (
    taxi_pdf
    .groupBy('PULocationID')
    .agg(
        F.count('*').alias('trips'),
        F.round(F.sum('fare_amount'), 2).alias('revenue'),
        F.round(F.avg('tip_amount'), 2).alias('avg_tip'),
    )
    .orderBy(F.desc('revenue'))
)
zone_revenue.show(5, truncate=False)


## Persist lineage (Parquet — primary)

Parquet writes produce the clearest Spline graphs: `Project` for
`trip_minutes`, `Aggregate` for `revenue`. Inspect these events first
in the UI.


In [ ]:
(trip_durations.write
 .mode('overwrite')
 .parquet(f'{PARQUET_SINK}/trip_durations'))

(zone_revenue.write
 .mode('overwrite')
 .parquet(f'{PARQUET_SINK}/zone_revenue'))

print('wrote trip_durations and zone_revenue (parquet)')


## Optional: Iceberg tables (analytics artifact)

Iceberg `CreateTableAsSelect` wraps the Spark plan in extra nodes.
Use these tables for downstream SQL, but prefer the Parquet events
above when exploring column lineage in Spline UI.


In [ ]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS local.taxi')

(trip_durations.write
 .format('iceberg')
 .mode('overwrite')
 .saveAsTable('local.taxi.trip_durations'))

(zone_revenue.write
 .format('iceberg')
 .mode('overwrite')
 .saveAsTable('local.taxi.zone_revenue'))

print('wrote trip_durations and zone_revenue (iceberg)')


## Inspect lineage (Consumer API)

Text fallback when the graph UI feels opaque. Also open
<http://localhost:9090> and follow the cheat sheet above.

Consumer API docs: <http://localhost:8080/docs/consumer.html>


In [ ]:
from _shared.lineage_inspect import list_recent_events, inspect_event_column

list_recent_events()
inspect_event_column('trip_durations', 'trip_minutes')
inspect_event_column('zone_revenue', 'revenue')
